In [1]:
import numpy as np

# Modulo Labo1

In [ ]:
def error(x, y):
    x = np.float64(x)
    y = np.float64(y)

    error = abs(x - y)

    return error


In [ ]:
def error_relativo(x, y):
    x = np.float64(x)
    y = np.float64(y)

    numerador = abs(x - y)
    denominador = abs(x)

    error_relativo = numerador/denominador

    return error_relativo

In [ ]:
def matricesIguales(A, B):

    if len(A) != len(B) or len(A[0]) != len(B[0]):
        return False

    # Aca suponemos que las dimensiones son iguales
    for n in range (len(A)):
        # no necesariamente es cuadrada
        for m in range(len(A[0])):
            # if A[n][m] != B[n][m]:
            # Con el distinto no me pasa por lo que lo panteo con error
            if abs (A[n][m] - B[n][m]) > 1e-08:
                return False

    return True

## Test


In [ ]:
def sonIguales(x,y,atol=1e-08):
    return np.allclose(error(x,y),0,atol=atol)

assert(not sonIguales(1,1.1))
assert(sonIguales(1,1 + np.finfo('float64').eps))
assert(not sonIguales(1,1 + np.finfo('float32').eps))
assert(not sonIguales(np.float16(1),np.float16(1) + np.finfo('float32').eps))
assert(sonIguales(np.float16(1),np.float16(1) + np.finfo('float16').eps,atol=1e-3))

assert(np.allclose(error_relativo(1,1.1),0.1))
assert(np.allclose(error_relativo(2,1),0.5))
assert(np.allclose(error_relativo(-1,-1),0))
assert(np.allclose(error_relativo(1,-1),2))

assert(matricesIguales(np.diag([1,1]),np.eye(2)))
assert(matricesIguales(np.linalg.inv(np.array([[1,2],[3,4]]))@np.array([[1,2],[3,4]]),np.eye(2)))
assert(not matricesIguales(np.array([[1,2],[3,4]]).T,np.array([[1,2],[3,4]])))

# Modulo Labo 2

In [ ]:
def rota(theta):
    # Para esta matriz tenemos que la forma es [[cos, sin], [-sin,cos]]
    # usamos las funciones trigonometricas de numpy

    R = np.array([[np.cos(theta), -np.sin(theta)],
                 [np.sin(theta), np.cos(theta)]])

    return R

In [ ]:
def escala(s):
    n = len(s)
    S = np.zeros((n,n))

    for i in range(len(s)):
        S[i][i] = s[i]

    return S

In [6]:
def multiplicacionMatricial(A,B):

    if len(A[0]) !=  len(B):
        return None

    res = np.zeros((len(A), len(B[0])), dtype = float)
    suma = 0

    for i in range(len(A)):
        for j in range(len(B[0])):
            # Este k puede iterar tanto en la len de las columans de A
            # como en la len de las filas de B
            for k in range(len(A[0])):
                elem_A = A[i][k]
                elem_B = B[k][j]
                suma += elem_A * elem_B

            res [i][j] = suma
            suma = 0

    return res



In [ ]:
def rota_y_escala(theta, s):
    # ojo que para que la matriz sea de 2*2, s tiene que tener len(s) = 2
    if len(s) != 2:
        return None
    R = rota(theta)
    S = escala(s)

    # Si primero rota y despues escala es RoS
    res = multiplicacionMatricial(S,R)

    return res



In [ ]:
def afin(theta,s,b):
    if len(s) != 2 or len(b) != 2:
        return None

    R_aux = rota(theta)
    S_aux = escala(s)

    multiplicacion = multiplicacionMatricial(S_aux, R_aux)

    res = np.zeros((3,3))

    for i in range(2):
        for j in range(2):
            res[i][j] = multiplicacion[i][j]

    for i in range(len(b)):
        res[i][2] = b[i]

    res[2][2] = 1

    return res



In [ ]:
def multiplicacionMatrizVector(A, v):

    if len(A[0]) != len(v):
        return None

    res = np.zeros(len(A))

    for i in range(len(A[0])):
        suma = 0
        for k in range(len(v)):
            suma += A[i][k] * v[k]

        res[i] = suma

    return res




In [ ]:
A = np.array([[1,2,1],[1,1,0], [2,1,4]])
v = np.array([3,1,1])
multiplicacionMatrizVector(A, v)

array([ 6.,  4., 11.])

In [ ]:
def trans_afin(v, theta, s, b):
    A = afin(theta, s, b)

    nuevo_v = np.zeros(3)

    for i in range(len(v)):
        nuevo_v[i] = v[i]
    nuevo_v[2] = 1
    A_v = multiplicacionMatrizVector(A, nuevo_v)

    res = A_v[:2]


    return res

## Test

In [ ]:

import numpy as np
# from lab2 import rota, escala, rota_y_escala, afin, trans_afin


# --- Tests extraídos del enunciado ---

print("\nIniciando serie de tests del Laboratorio 2...")

# Tests para rota
assert(np.allclose(rota(0), np.eye(2)))
assert(np.allclose(rota(np.pi/2), np.array([[0, -1],[1, 0]])))
assert(np.allclose(rota(np.pi), np.array([[-1, 0],[0, -1]])))

# Tests para escala
assert(np.allclose(escala([2,3]), np.array([[2,0],[0,3]])))
assert(np.allclose(escala([1,1,1]), np.eye(3)))
assert(
    np.allclose(escala([0.5,0.25]), np.array([[0.5,0],[0,0.25]]))
)

# Tests para rota_y_escala
assert(
    np.allclose(rota_y_escala(0,[2,3]), np.array([[2,0],[0,3]]))
)
assert(np.allclose(
    rota_y_escala(np.pi/2,[1,1]), np.array([[0,-1],[1,0]])
))
assert(np.allclose(
    rota_y_escala(np.pi,[2,2]), np.array([[-2,0],[0,-2]])
))

# Tests para afin
assert(np.allclose(
    afin(0,[1,1],[1,2]),
    np.array([[1,0,1],
              [0,1,2],
              [0,0,1]])))
assert(np.allclose(afin(np.pi/2,[1,1],[0,0]),
    np.array([[0,-1,0],
              [1,0,0],
              [0,0,1]])))
assert(np.allclose(afin(0,[2,3],[1,1]),
    np.array([[2,0,1],
              [0,3,1],
              [0,0,1]])))

# Tests para trans_afina

assert(np.allclose(
    trans_afin(np.array([1,0]), np.pi/2,[1,1],[0,0]),
    np.array([0,1])
))
assert(np.allclose(
    trans_afin(np.array([1,1]), 0,[2,3],[0,0]),
    np.array([2,3])
))
assert(np.allclose(
    trans_afin(np.array([1,0]), np.pi/2,[3,2],[4,5]),
    np.array([4,7])
))

print("¡Todos los tests del Laboratorio 2 han pasado exitosamente!")


Iniciando serie de tests del Laboratorio 2...
¡Todos los tests del Laboratorio 2 han pasado exitosamente!


# Modulo Labo 3

In [2]:
def norma(x, p):
    # x es el vector y p es la norma
    for i in range(len(x)):
        x[i] = abs(x[i])

    if p == 'inf':
        return max(x)

    suma = 0
    for elem in x:
        suma += elem**p

    raiz = suma**(1/p)

    return raiz


In [3]:
def normaliza(X, p):
    # X es una lista de vectores
    Y = []
    for vector in X:
        norma_v = norma(vector, p)
        y = np.zeros(len(vector))
        for i in range(len(vector)):
            elem = vector[i]
            y[i] = elem/norma_v

        Y.append(y)

    return Y

In [4]:
def normaExacta(A, p=[1, 'inf']):

    # inf
    suma_filas = []
    # 1
    suma_columnas = []

    for i in range(len(A)):
        suma = 0
        for j in range(len(A[0])):
            suma += abs(A[i][j])

        suma_filas.append(suma)

    for i in range(len(A[0])):
        suma = 0
        for j in range(len(A)):
            suma += abs(A[j][i])

        suma_columnas.append(suma)


    if type(p) == list:
        return [max(suma_columnas), max(suma_filas)]
    elif p == 1:
        return max(suma_columnas)
    elif p == 'inf':
        return max(suma_filas)
    else:
        return None

In [7]:
def condMC(A,p):
  inversa = np.linalg.inv(A)

  # Pongo 10000 porque es el valor que esta en los test
  # Es una multiplicacion de numeros no es matricial
  cond = normaMatMC(A,p,p,10000)[0] * normaMatMC(inversa, p, p, 10000)[0]

  return cond

In [9]:
def condExacto(A,p):
  inversa = np.linalg.inv(A)

  cond = normaExacta(A,p) * normaExacta(inversa, p)

  return cond

## Test


In [12]:
# Tests L03-Normas

# Tests norma
assert(np.allclose(norma(np.array([1,1]),2),np.sqrt(2)))
assert(np.allclose(norma(np.array([1]*10),2),np.sqrt(10)))
# assert(norma(np.random.rand(10),2)<=np.sqrt(10))
# assert(norma(np.random.rand(10),2)>=0)

# Tests normaliza
# Tests normaliza
for x in normaliza([np.array([1]*k) for k in range(1,11)],2):
    assert(np.allclose(norma(x,2),1))
for x in normaliza([np.array([1]*k) for k in range(2,11)],1):
    print(not np.allclose(norma(x,2),1) )
for x in normaliza([np.random.rand(k) for k in range(1,11)],'inf'):
    assert( np.allclose(norma(x,'inf'),1) )


# Tests normaExacta

assert(np.allclose(normaExacta(np.array([[1,-1],[-1,-1]]),1),2))
assert(np.allclose(normaExacta(np.array([[1,-2],[-3,-4]]),1),6))
assert(np.allclose(normaExacta(np.array([[1,-2],[-3,-4]]),'inf'),7))
assert(normaExacta(np.array([[1,-2],[-3,-4]]),2) is None)
assert(normaExacta(np.random.random((10,10)),1)<=10)
assert(normaExacta(np.random.random((4,4)),'inf')<=4)
'''
# Test normaMC

nMC = normaMatMC(A=np.eye(2),q=2,p=1,Np=100000)
assert(np.allclose(nMC[0],1,atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),0,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),0,atol=1e-3))

nMC = normaMatMC(A=np.eye(2),q=2,p='inf',Np=100000)
assert(np.allclose(nMC[0],np.sqrt(2),atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) and np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))

A = np.array([[1,2],[3,4]])
nMC = normaMatMC(A=A,q='inf',p='inf',Np=1000000)
assert(np.allclose(nMC[0],normaExacta(A,'inf'),rtol=2e-1))
'''
# Test condMC

A = np.array([[1,1],[0,1]])
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaMatMC(A,2,2,10000)
normaA_ = normaMatMC(A_,2,2,10000)
condA = condMC(A,2,10000)
assert(np.allclose(normaA[0]*normaA_[0],condA,atol=1e-3))

A = np.array([[3,2],[4,1]])
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaMatMC(A,2,2,10000)
normaA_ = normaMatMC(A_,2,2,10000)
condA = condMC(A,2,10000)
assert(np.allclose(normaA[0]*normaA_[0],condA,atol=1e-3))

# Test condExacta

A = np.random.rand(10,10)
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaExacta(A,1)
normaA_ = normaExacta(A_,1)
condA = condExacto(A,1)
assert(np.allclose(normaA*normaA_,condA))

A = np.random.rand(10,10)
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaExacta(A,'inf')
normaA_ = normaExacta(A_,'inf')
condA = condExacto(A,'inf')
assert(np.allclose(normaA*normaA_,condA))


True
True
True
True
True
True
True
True
True


NameError: name 'normaMatMC' is not defined


np.cos()


np.sen()


np.eye()


np.shape()


np.zeros()


np.copy()


np.ones()


np.ndim()


np.arrange()


np.linspace()


np.array()


np.reshape()


Funciones del submódulo np.random que sirvan para generar números pseudo-aleatorios.


Operaciones de slicing

